# Kuiper kernel benchmarksEach table compares the JIT-dispatched verified Kuiper kernels (`kuipy.run`) andthe unverified reference kernels (`kuipy.unverified`) against stock PyTorch, onthe shapes of a Qwen2.5-0.5B decode step.Times are us/call, `rel-err` is the relative Frobenius norm against the `ref` column.

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

import kuipy
from kuipy import unverified
from kuipy.benchmarking import bench_matrix

aten = torch.ops.aten
DEV = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Qwen2.5-0.5B-Instruct, decoding at batch 256.
HID, NH, NKV, HEAD_DIM = 896, 14, 2, 64
INTER, VOCAB, BATCH = 4864, 151936, 256
SCALE = HEAD_DIM ** -0.5
ALPHA, BETA = 0.75, 1.5

_g = torch.Generator(device=DEV).manual_seed(0)

def rand(*shape, dtype=torch.bfloat16):
    return torch.randn(*shape, device=DEV, dtype=dtype, generator=_g) * 0.1

torch.cuda.get_device_name(0)

## mm`C = A @ B`. The unverified GEMMs are addmm-shaped, so they show up in the next section.

In [ ]:
MM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("gate_proj",   BATCH, HID,   INTER),
    ("up_proj",     BATCH, HID,   INTER),
    ("down_proj",   BATCH, INTER, HID),
    ("lm_head",     BATCH, HID,   VOCAB),
    ("square_4096", 4096,  4096,  4096),
]

def mm_inputs(dtype):
    return lambda M, K, N: ((rand(M, K, dtype=dtype), rand(K, N, dtype=dtype)), {})

MNK = lambda M, K, N: (M, K, N)
GEMM_FLOPS = lambda M, K, N: 2 * M * N * K

bench_matrix(MM_CASES, mm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to"))],
             torch.mm, flops=GEMM_FLOPS)

In [ ]:
bench_matrix(MM_CASES, mm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to")),
              ("tc2d", kuipy.run(aten.mm.default, impl="tc2d",
                                 acc_dtype=torch.float16))],
             torch.mm, flops=GEMM_FLOPS)

## addmm`D = beta*C + alpha*(A @ B)`: the full epilogue, which the bias-free `mm` path never hits.`gemm_pipe` is fp16-only and `gemm_hacky_epilogue` bf16-only, hence the two tables.

In [ ]:
GEMM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
    ("square_4096", 4096,  4096,  4096),
]

def addmm_inputs(dtype):
    return lambda M, K, N: ((rand(M, N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(K, N, dtype=dtype)), {"beta": BETA, "alpha": ALPHA})

bench_matrix(GEMM_CASES, addmm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("hacky_epilogue", unverified.gemm_hacky_epilogue)],
             torch.addmm, flops=GEMM_FLOPS)

In [ ]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("gemm_pipe", unverified.gemm_pipe)],
             torch.addmm, flops=GEMM_FLOPS)

## sdpaThe Kuiper kernel's mask is a dense `(B, Hq, Sq, Sk)` tensor -- a Kuiper tlayout is aninjection, so there is no broadcast layout to instantiate it with -- so the mask ismaterialised and handed to every contender for fairness.

In [ ]:
_kuiper_sdpa = kuipy.run(aten._scaled_dot_product_efficient_attention.default)

def kuiper_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    return _kuiper_sdpa(q, k, v, attn_mask, False, 0.0, is_causal, scale=scale)[0]

def cudnn_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                              is_causal=is_causal, scale=scale,
                                              enable_gqa=True)

def attn_flops(sq, sk):
    return 4 * BATCH * NH * sq * sk * HEAD_DIM

In [ ]:
DECODE_CASES = [(f"ctx_{c}", 1, c) for c in (128, 512, 1024, 16384)]

def decode_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"attn_mask": torch.zeros(BATCH, NH, sq, sk, device=DEV,
                                      dtype=torch.bfloat16),
             "scale": SCALE})

bench_matrix(DECODE_CASES, decode_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("manual_extract", unverified.flash_attn_manual_extract),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

In [ ]:
# Prefill: full self-attention, is_causal, no explicit mask -- which the Kuiper
# kernel cannot express (see above), so only the unverified kernels compete.
PREFILL_CASES = [(f"seq_{s}", s, s) for s in (128, 512)]

def prefill_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"is_causal": True, "scale": SCALE})

bench_matrix(PREFILL_CASES, prefill_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)